# BoTorch Acquisition Functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/instadeepai/alf/blob/main/tutorials/extending_base_classes/botorch_acquisition_functions.ipynb)

This tutorial shows how to use `BoTorchAcquisition` from
[`alf_tools.optimizer.acquisition_functions`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/botorch_acquisition/).

`BoTorchAcquisition` is a unified wrapper around BoTorch's analytic and Monte Carlo
acquisition functions. The `acquisition_type` argument selects the strategy; no
separate config object or registry is needed. The class supports two modes:

1. **Discrete mode** — score a provided pool of candidates (demonstrated here)
2. **Continuous mode** — call BoTorch's `optimize_acqf` directly (requires `bounds`)

It accepts **either** a native BoTorch `Model` **or** an ALF `BaseModel` and inserts a
`BotorchModelWrapper` automatically when needed. Here we use ALF's `GPModel` as the
surrogate to demonstrate the full integration.

`best_f` and `X_baseline` are derived **internally** from the training data — the user
must not pass them. Invalid `acquisition_type` values raise `ValueError` at construction
time.

## Setup

Run the cell below to install ALF and this tutorial's dependencies — **no repository clone required**, so it works in a fresh environment or on Google Colab.

- Already set up a dev environment from a clone (`uv sync`)? You can **skip the install cell**.
- To run on a **GPU**, uncomment the GPU line in the install cell.

For all installation options, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

In [ ]:
# Install ALF + this tutorial's dependencies — no clone needed.
# (Skip this cell if you are already running from a cloned repo via `uv sync`.)
%pip install "git+https://github.com/instadeepai/alf.git#subdirectory=tools" matplotlib
# GPU (optional): run this AFTER the line above to switch PyTorch to a CUDA build.
# %pip install torch --index-url https://download.pytorch.org/whl/cu128
# Once ALF is on PyPI this simplifies to e.g. `%pip install alf_tools` (no git URL).

## 1. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from alf_core.dataclasses.candidate import Candidate, Modality
from alf_core.dataclasses.labelled_candidates import LabelledCandidates
from alf_tools.models import FeaturizerConfig, GPModel, GPModelConfig, GPTrainConfig
from alf_tools.optimizer.acquisition_functions import BoTorchAcquisition

print("Imports successful")

## 2. Train a GP Surrogate

We fit a `GPModel` to a synthetic 1D function.  The `"precomputed"` featuriser
accepts `Candidate.data` values that are already `np.ndarray` or `torch.Tensor`,
so no custom featuriser function is needed for tabular data.

In [ ]:
np.random.seed(42)
X_MIN, X_MAX = 0.0, 2 * np.pi
N_TRAIN = 20


def true_fn(x):
    return np.sin(x) + 0.5 * np.sin(3 * x)


x_train = np.sort(np.random.uniform(X_MIN, X_MAX, N_TRAIN))
y_train = true_fn(x_train) + np.random.randn(N_TRAIN) * 0.1


def make_candidates(x_array):
    """Wrap 1D points as ALF Candidates with precomputed numpy features."""
    return [Candidate(data=np.array([xi]), modality=Modality.TABULAR) for xi in x_array]


train_data = LabelledCandidates(make_candidates(x_train), y_train)
search_candidates = make_candidates(np.linspace(X_MIN, X_MAX, 100))

gp = GPModel(
    model_config=GPModelConfig(kernel_type="matern", matern_nu=2.5),
    train_config=GPTrainConfig(num_iterations=100, log_frequency=100),
    featurizer_config=FeaturizerConfig(featurizer_type="precomputed"),
)
gp.train(train_data)

print(f"GP trained on {N_TRAIN} points")
print(f"Learned hyperparameters: {gp.get_hyperparameters()}")

## 3. `BoTorchAcquisition`

`BoTorchAcquisition` implements the ALF `AcquisitionFunction` interface:
`__call__(candidates, state) → LabelledCandidates`. Pass `acquisition_type` to
select the strategy — no config object or registry needed.

> **`best_f` and `X_baseline` are internal.** They are derived from
> `state.dataset.train_dataset` automatically — do **not** pass them.

> **In a real `DesignTask`**, `state` is the `State` object produced by
> `task.setup()`. Below we use a minimal stub (with `surrogate` and `dataset`
> attributes) to keep the example self-contained.

In [ ]:
# Minimal stub that mirrors the real State used in a DesignTask.
# BoTorchAcquisition needs state.surrogate.model (the GP) and
# state.dataset.train_dataset (the LabelledCandidates used to derive best_f
# and X_baseline internally).
class _MockSurrogate:
    def __init__(self, model):
        self.model = model


class _MockDataset:
    def __init__(self, train_dataset):
        self.train_dataset = train_dataset


class _MockState:
    def __init__(self, model, train_dataset):
        self.surrogate = _MockSurrogate(model)
        self.dataset = _MockDataset(train_dataset)


state = _MockState(gp, train_data)

# -- Log Expected Improvement --
ei_fn = BoTorchAcquisition(acquisition_type="log_expected_improvement")
ei_result = ei_fn(search_candidates, state)

# -- Upper Confidence Bound --
ucb_fn = BoTorchAcquisition(acquisition_type="upper_confidence_bound", beta=2.0)
ucb_result = ucb_fn(search_candidates, state)

# -- Probability of Improvement --
poi_fn = BoTorchAcquisition(acquisition_type="probability_of_improvement")
poi_result = poi_fn(search_candidates, state)

# -- Log q-Noisy Expected Improvement (X_baseline derived internally) --
lnei_fn = BoTorchAcquisition(acquisition_type="log_noisy_expected_improvement")
lnei_result = lnei_fn(search_candidates, state)

print(f"EI   scores: [{ei_result.labels.min():.4f}, {ei_result.labels.max():.4f}]")
print(f"UCB  scores: [{ucb_result.labels.min():.4f}, {ucb_result.labels.max():.4f}]")
print(f"POI  scores: [{poi_result.labels.min():.4f}, {poi_result.labels.max():.4f}]")
print(f"LNEI scores: [{lnei_result.labels.min():.4f}, {lnei_result.labels.max():.4f}]")
print(f"\nAll return LabelledCandidates: {type(ei_result).__name__}")

Switching acquisition functions is a one-line change — just pass a different
`acquisition_type`. `best_f` and `X_baseline` are derived automatically from the
model's training data, so no manual bookkeeping is required.

`log_expected_improvement` uses BoTorch's `LogExpectedImprovement` for numerical
stability — scores are in log-space (≤ 0), where higher (less negative) values
indicate more expected improvement. UCB (`mean + β·std`) balances exploitation
(high mean) and exploration (high uncertainty), with `beta` controlling the trade-off.

In [ ]:
preds = gp.predict(search_candidates)
x_search = np.linspace(X_MIN, X_MAX, 100)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

# Top: GP prediction
ax = axes[0]
ax.plot(x_search, true_fn(x_search), "k--", linewidth=1.2, alpha=0.6, label="True fn")
ax.scatter(x_train, y_train, s=50, color="black", zorder=5, label="Training data")
ax.plot(x_search, preds.means, color="#7b2d8b", linewidth=2, label="GP mean")
ax.fill_between(
    x_search,
    preds.means - 2 * np.sqrt(preds.variances),
    preds.means + 2 * np.sqrt(preds.variances),
    alpha=0.2,
    color="#7b2d8b",
    label="±2σ",
)
ax.set_ylabel("y")
ax.set_title("GP Surrogate (Matérn-2.5)")
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)

# Middle: log-EI
ax = axes[1]
ax.plot(x_search, ei_result.labels, color="#e74c3c", linewidth=2)
ax.fill_between(x_search, ei_result.labels.min(), ei_result.labels, alpha=0.2, color="#e74c3c")
ax.set_ylabel("log-EI score")
ax.set_title("Log Expected Improvement (log-space, ≤ 0; peaks where improvement is likely)")
ax.grid(True, alpha=0.3)

# Bottom: UCB
ax = axes[2]
ax.plot(x_search, ucb_result.labels, color="#2980b9", linewidth=2)
ax.fill_between(x_search, ucb_result.labels.min(), ucb_result.labels, alpha=0.2, color="#2980b9")
ax.set_ylabel("UCB score")
ax.set_xlabel("x")
ax.set_title("Upper Confidence Bound (β=2.0; balances mean and uncertainty)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Validation

`BoTorchAcquisition` validates `acquisition_type` at construction time so mistakes
surface immediately:

- **Unknown `acquisition_type`** → `ValueError`

In [ ]:
# -- Unknown acquisition_type --
try:
    BoTorchAcquisition(acquisition_type="nonexistent_acq")
except ValueError as exc:
    print(f"Unknown type caught: {exc}")

## Key Points

- **Single interface**: `BoTorchAcquisition(acquisition_type=...)` covers all supported
  strategies — no registry or config object needed.
- **Valid `acquisition_type` values**: `"qEI"`, `"qLogEI"`, `"qNEI"`, `"qUCB"`,
  `"log_expected_improvement"`, `"upper_confidence_bound"`,
  `"probability_of_improvement"`, `"log_noisy_expected_improvement"`.
- **Internal bookkeeping**: `best_f` and `X_baseline` are derived from the model's
  training data automatically — do not pass them.
- **Model-agnostic**: Accepts either a native BoTorch `Model` or an ALF `BaseModel` —
  `BotorchModelWrapper` is inserted automatically.
- **ALF-compatible**: Implements `AcquisitionFunction` (`__call__(candidates, state) →
  LabelledCandidates`); works wherever ALF expects an `AcquisitionFunction`.
- **Two modes**: Discrete candidate scoring (non-empty pool) or continuous
  `optimize_acqf` optimisation (empty pool, requires `bounds`).
- **Eager validation**: Invalid `acquisition_type` raises `ValueError`
- **Variance required**: Models must provide variance estimates. Deterministic models
  (e.g. `CNNModel`) cannot be used with analytic BoTorch acquisitions and will raise a
  `ValueError` with a descriptive message.

See `tools/alf_tools/optimizer/acquisition_functions/botorch_acquisition.py` for the full
implementation.